In [ ]:
import os
import shutil
import random
from glob import glob

# --- 設定 ---
SOURCE_DIR = "labelImg-master\\trainimg"
TARGET_BASE_DIR = "yolo_data"
CLASSES_FILE = os.path.join(SOURCE_DIR, "classes.txt")

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

SEED = 100
random.seed(SEED)

IMAGE_EXT = ".jpg"

# --- 1️⃣ 收集每張圖片的「所有類別」 ---
print("🔍 正在分析多類別分佈（YOLO正確模式）...")

image_files = glob(os.path.join(SOURCE_DIR, "*" + IMAGE_EXT))
image_classes = {}

for img_path in image_files:
    name = os.path.basename(img_path).replace(IMAGE_EXT, "")
    txt_path = os.path.join(SOURCE_DIR, name + ".txt")

    classes = set()

    if os.path.exists(txt_path):
        with open(txt_path, 'r') as f:
            for line in f:
                if line.strip():
                    cls = line.split()[0]
                    classes.add(cls)

    # 沒有標註 → 當成 empty 類
    if not classes:
        classes.add("empty")

    image_classes[name] = classes

# --- 2️⃣ Stratified Split（多類別） ---
print("⚖️ 正在進行分層切分...")

train_names, val_names, test_names = set(), set(), set()

all_names = list(image_classes.keys())
random.shuffle(all_names)

n = len(all_names)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_names = set(all_names[:n_train])
val_names = set(all_names[n_train:n_train+n_val])
test_names = set(all_names[n_train+n_val:])

print(f"✅ 切分完成")
print(f"Train: {len(train_names)}")
print(f"Val:   {len(val_names)}")
print(f"Test:  {len(test_names)}")

# --- 4️⃣ 建立資料夾 ---
def setup_dirs(split):
    base = os.path.join(TARGET_BASE_DIR, split)
    if os.path.exists(base):
        shutil.rmtree(base)
    os.makedirs(os.path.join(base, "images"), exist_ok=True)
    os.makedirs(os.path.join(base, "labels"), exist_ok=True)

splits = {
    "train": train_names,
    "val": val_names,
    "test": test_names
}

empty_txt_count = 0

# --- 5️⃣ 複製資料 ---
for split, names in splits.items():
    setup_dirs(split)

    img_dir = os.path.join(TARGET_BASE_DIR, split, "images")
    lbl_dir = os.path.join(TARGET_BASE_DIR, split, "labels")

    print(f"🚀 複製 {split} 資料...")

    for name in names:
        src_img = os.path.join(SOURCE_DIR, name + IMAGE_EXT)
        dst_img = os.path.join(img_dir, name + IMAGE_EXT)

        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)

        src_lbl = os.path.join(SOURCE_DIR, name + ".txt")
        dst_lbl = os.path.join(lbl_dir, name + ".txt")

        if os.path.exists(src_lbl) and os.path.getsize(src_lbl) > 0:
            shutil.copy(src_lbl, dst_lbl)
        else:
            open(dst_lbl, 'w').close()
            empty_txt_count += 1

# --- 6️⃣ 產生 data.yaml ---
print("\n📝 產生 data.yaml...")

class_names = []
if os.path.exists(CLASSES_FILE):
    with open(CLASSES_FILE, 'r', encoding='utf-8') as f:
        class_names = [line.strip() for line in f if line.strip()]

yaml_content = f"""# YOLO Dataset
path: {os.path.abspath(TARGET_BASE_DIR)}
train: train/images
val: val/images
test: test/images

nc: {len(class_names)}
names: {class_names}
"""

with open(os.path.join(TARGET_BASE_DIR, "data.yaml"), "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"✅ 完成！空標註檔數量: {empty_txt_count}")

In [1]:
#分析切分後的類別分佈
import os
import yaml
import pandas as pd

# --- 設定路徑 ---
YOLO_DATA_DIR = 'yolo_data'  # 這裡請填入你的 YOLO 資料夾路徑
DATA_YAML_PATH = os.path.join(YOLO_DATA_DIR, 'data.yaml')

# --- 函數：讀取並統計標註 ---
def count_labels(label_dir, class_names):
    """
    遍歷指定資料夾內的 .txt 標註檔，並統計:
    1. 每個類別的物件數量
    2. 純背景圖片 (空標註檔) 的數量
    """
    counts = {name: 0 for name in class_names}
    total_files = 0
    total_objects = 0
    background_files = 0 
    
    if not os.path.exists(label_dir):
        print(f"⚠️ 警告：找不到路徑 {label_dir}。跳過此分割。")
        return counts, 0, 0, 0

    for filename in os.listdir(label_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(label_dir, filename)
            total_files += 1
            
            try:
                # 檢查是否為空檔案 (0 bytes)
                if os.path.getsize(file_path) == 0:
                    background_files += 1
                    continue

                has_objects = False
                with open(file_path, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        
                        has_objects = True 
                        
                        try:
                            # YOLO 格式: <class_id> <x> <y> <w> <h>
                            class_index = int(line.split()[0])
                            
                            if 0 <= class_index < len(class_names):
                                class_name = class_names[class_index]
                                counts[class_name] += 1
                                total_objects += 1
                        except (ValueError, IndexError):
                            continue 
                
                if not has_objects:
                    background_files += 1
                                    
            except Exception as e:
                print(f"❌ 錯誤：讀取標註檔 {filename} 失敗: {e}")
                continue

    return counts, total_files, total_objects, background_files


# --- 主程式流程 ---
def analyze_split_distribution():
    # 1. 載入類別名稱
    try:
        with open(DATA_YAML_PATH, 'r', encoding='utf-8') as f:
            data_yaml = yaml.safe_load(f)
        
        class_names = data_yaml.get('names')
        if not class_names:
            raise ValueError("在 data.yaml 中找不到 'names' 列表。")
        
        print(f"✅ 成功讀取 {len(class_names)} 個類別: {class_names}")
    
    except FileNotFoundError:
        print(f"❌ 錯誤：找不到 data.yaml 檔案: {DATA_YAML_PATH}")
        return
    except Exception as e:
        print(f"❌ 錯誤：解析 data.yaml 失敗: {e}")
        return

    # 2. 統計各個分割集
    splits = ['train', 'val', 'test']
    results = {}

    for s in splits:
        label_dir = os.path.join(YOLO_DATA_DIR, s, 'labels')
        counts, files, objects, bg = count_labels(label_dir, class_names)
        results[s] = {
            'counts': counts,
            'files': files,
            'objects': objects,
            'bg': bg
        }

    # 3. 整理資料夾物件分佈表格
    df_data = {
        'Train Objects': results['train']['counts'],
        'Val Objects': results['val']['counts'],
        'Test Objects': results['test']['counts']
    }
    df = pd.DataFrame(df_data).T 
    df['Total Objects'] = df.sum(axis=1)
    
    # 4. 輸出總結
    print("\n" + "="*60)
    print("--- 訓練 / 驗證 / 測試資料集統計總結 ---")
    print("="*60)
    
    for s in splits:
        icon = "📘" if s == 'train' else ("📙" if s == 'val' else "📗")
        name = s.capitalize()
        res = results[s]
        print(f"{icon} {name} Set:")
        print(f"   - 總圖片數:      {res['files']}")
        print(f"   - 有物件圖片:    {res['files'] - res['bg']}")
        print(f"   - 純背景圖片:    {res['bg']} (負樣本)")
        print(f"   - 標註物件總數:  {res['objects']}")
        print("-" * 40)
    
    # 輸出詳細的類別分佈表
    print("\n📦 類別物件數量分佈 (Object Counts)")
    print(df.to_markdown(numalign="left", stralign="left"))

    # 5. 檢查分佈平衡性 (比例分析)
    print("\n💡 平衡性檢查 (以 Train 為基準的比例)")
    print(f"{'Class Name':<15} | {'Train:Val':<10} | {'Train:Test':<10}")
    print("-" * 45)

    def get_ratio_str(train_val, target_val):
        if target_val == 0:
            return "Inf" if train_val > 0 else "0.0"
        return f"{train_val / target_val:.1f}"

    # 背景圖比例
    bg_v = get_ratio_str(results['train']['bg'], results['val']['bg'])
    bg_t = get_ratio_str(results['train']['bg'], results['test']['bg'])
    print(f"{'[Background]':<15} | {bg_v:<10} | {bg_t:<10}")

    for class_name in class_names:
        tr_c = results['train']['counts'].get(class_name, 0)
        va_c = results['val']['counts'].get(class_name, 0)
        te_c = results['test']['counts'].get(class_name, 0)
        
        ratio_v = get_ratio_str(tr_c, va_c)
        ratio_t = get_ratio_str(tr_c, te_c)

        print(f"{class_name:<15} | {ratio_v:<10} | {ratio_t:<10}")
    
    print("\n(註：若比例設定為 8:1:1，建議比例約為 8.0)")     
    print("="*60)


if __name__ == "__main__":
    analyze_split_distribution()

✅ 成功讀取 3 個類別: ['level_1', 'level_2', 'level_3']

--- 訓練 / 驗證 / 測試資料集統計總結 ---
📘 Train Set:
   - 總圖片數:      960
   - 有物件圖片:    960
   - 純背景圖片:    0 (負樣本)
   - 標註物件總數:  5919
----------------------------------------
📙 Val Set:
   - 總圖片數:      120
   - 有物件圖片:    120
   - 純背景圖片:    0 (負樣本)
   - 標註物件總數:  694
----------------------------------------
📗 Test Set:
   - 總圖片數:      120
   - 有物件圖片:    120
   - 純背景圖片:    0 (負樣本)
   - 標註物件總數:  817
----------------------------------------

📦 類別物件數量分佈 (Object Counts)
|               | level_1   | level_2   | level_3   | Total Objects   |
|:--------------|:----------|:----------|:----------|:----------------|
| Train Objects | 2772      | 1282      | 1865      | 5919            |
| Val Objects   | 299       | 153       | 242       | 694             |
| Test Objects  | 348       | 160       | 309       | 817             |

💡 平衡性檢查 (以 Train 為基準的比例)
Class Name      | Train:Val  | Train:Test
---------------------------------------------
[Background]    | 0.0

In [ ]:
# 計算尺寸統計
import os
import yaml
import glob
from PIL import Image
import pandas as pd
import numpy as np
import sys

# --- 參數設定 ---
YOLO_DATA_DIR = 'yolo_data'  # 這裡請填入你的 YOLO 資料夾路徑
DATA_YAML_PATH = os.path.join(YOLO_DATA_DIR, 'data.yaml')

def load_config(yaml_path):
    """載入 data.yaml 並進行基本檢查"""
    try:
        with open(yaml_path, 'r', encoding='utf-8') as f:
            data_yaml = yaml.safe_load(f)
        
        if not isinstance(data_yaml, dict):
            print(f"❌ 錯誤：data.yaml 格式不正確，應該是字典格式，但讀取到 {type(data_yaml)}")
            return None
            
        return data_yaml
    except FileNotFoundError:
        print(f"❌ 錯誤：找不到 data.yaml 檔案於 {yaml_path}")
        return None
    except Exception as e:
        print(f"❌ 錯誤：無法解析 {yaml_path}。詳細: {e}")
        return None

def resolve_path(path_in_yaml, yaml_file_path):
    """
    智慧路徑解析：嘗試多種可能性來找到真實路徑
    1. 檢查是否為絕對路徑或相對於當前工作目錄存在的路徑
    2. 檢查是否相對於 data.yaml 檔案所在位置
    """
    if not path_in_yaml:
        return None

    # 1. 直接檢查 (適用於絕對路徑 或 相對於執行腳本的路徑)
    if os.path.exists(path_in_yaml):
        return path_in_yaml
    
    # 2. 相對於 yaml 檔案的位置 (YOLO 標準做法)
    yaml_dir = os.path.dirname(os.path.abspath(yaml_file_path))
    path_relative_to_yaml = os.path.join(yaml_dir, path_in_yaml)
    if os.path.exists(path_relative_to_yaml):
        return path_relative_to_yaml
        
    return None

def get_data_paths(config):
    """獲取圖片與標註檔的配對路徑"""
    paths = []
    
    for split in ['train', 'val', 'test']:
        images_rel = config.get(split)
        
        # 使用智慧路徑解析
        images_dir = resolve_path(images_rel, DATA_YAML_PATH)
        
        if images_dir:
            # 假設 labels 資料夾在 images 資料夾同層級的 ../labels
            # 這是 YOLO 的標準結構
            parent_dir = os.path.dirname(images_dir)
            labels_dir = os.path.join(parent_dir, 'labels')
            
            # 再次確認路徑存在
            if not os.path.exists(labels_dir):
                # 嘗試另一種結構：如果 images_dir 已經包含 images 字樣，直接替換
                if 'images' in images_dir:
                    labels_dir = images_dir.replace('images', 'labels')
            
            if os.path.exists(images_dir):
                # 搜尋圖片
                found_images = glob.glob(os.path.join(images_dir, '*'))
                valid_images = [img for img in found_images if img.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                
                if not valid_images:
                    print(f"⚠️ 警告: 在 {images_dir} 中找不到任何圖片。")
                    continue
                    
                for img_path in valid_images:
                    paths.append((img_path, labels_dir))
            else:
                print(f"⚠️ 警告: 無法定位 {split} 資料夾: {images_rel}")
        else:
            if images_rel:
                print(f"⚠️ 警告: 設定檔中的路徑無效: {images_rel}")
                
    return paths

def analyze_simple_resolution():
    
    # 檢查依賴
    try:
        import pandas as pd
        import numpy as np
    except ImportError:
        print("❌ 錯誤：需要安裝函式庫。請執行：pip install Pillow pandas numpy")
        sys.exit(1)
            
    print(f"📂 正在讀取設定: {DATA_YAML_PATH}")
    config = load_config(DATA_YAML_PATH)
    if config is None:
        return
        
    class_names = config.get('names', [])
    print(f"📋 偵測到類別: {class_names}")
    
    data_paths = get_data_paths(config)
    
    if not data_paths:
        print("❌ 未找到任何有效的圖片路徑。請檢查 yolo_data/data.yaml 的內容。")
        return

    print(f"✅ 成功載入 {len(data_paths)} 張圖片，開始分析...")

    results_list = []

    for img_path, labels_dir in data_paths:
        file_basename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(labels_dir, file_basename + '.txt')

        # 1. 讀取圖片尺寸
        try:
            with Image.open(img_path) as img:
                img_width, img_height = img.size
        except Exception:
            continue
        
        # 2. 讀取標註並轉換為像素尺寸
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    try:
                        parts = line.strip().split()
                        if len(parts) < 5: continue
                        
                        class_id = int(parts[0])
                        norm_w = float(parts[3])
                        norm_h = float(parts[4])
                        
                        bbox_width_px = norm_w * img_width
                        bbox_height_px = norm_h * img_height
                        bbox_area_px = bbox_width_px * bbox_height_px
                        
                        class_name = class_names[class_id] if 0 <= class_id < len(class_names) else f'Class_{class_id}'
                        
                        results_list.append({
                            'Class': class_name,
                            'Photo_Width_px': img_width,
                            'Photo_Height_px': img_height,
                            'BBox_Width_px': bbox_width_px,
                            'BBox_Height_px': bbox_height_px,
                            'BBox_Area_px': bbox_area_px,
                        })
                    except Exception:
                        continue

    if not results_list:
        print("❌ 雖然找到了圖片，但在對應的標註檔中沒有找到任何物件。")
        return

    df = pd.DataFrame(results_list)

    # --- 輸出統計結果 ---
    print("\n" + "="*50)
    print("--- 資料集解析度與標註尺寸統計 ---")
    print("="*50)

    # 1. 圖片解析度
    photo_df = df[['Photo_Width_px', 'Photo_Height_px']].drop_duplicates()
    photo_stats = photo_df.agg(['mean', 'median', 'min', 'max']).T
    
    print("\n🖼️ 圖片原始解析度 (Pixel)")
    print(photo_stats.to_markdown(numalign="left", stralign="left", floatfmt=".1f"))

    # 2. 標註框尺寸
    bbox_stats = df.groupby('Class')[['BBox_Width_px', 'BBox_Height_px', 'BBox_Area_px']].agg(['count', 'mean', 'median', 'min', 'max']).fillna(0)
    
    print("\n📦 標註框像素尺寸統計 (Pixel)")
    
    for class_name in bbox_stats.index:
        stats = bbox_stats.loc[class_name]
        count = int(stats['BBox_Width_px']['count'])
        
        print(f"\n--- 類別: {class_name} (總數: {count}) ---")
        
        output_data = {
            'Metric': ['Width', 'Height', 'Area'],
            'Mean': [stats['BBox_Width_px']['mean'], stats['BBox_Height_px']['mean'], stats['BBox_Area_px']['mean']],
            'Median': [stats['BBox_Width_px']['median'], stats['BBox_Height_px']['median'], stats['BBox_Area_px']['median']],
            'Min': [stats['BBox_Width_px']['min'], stats['BBox_Height_px']['min'], stats['BBox_Area_px']['min']],
            'Max': [stats['BBox_Width_px']['max'], stats['BBox_Height_px']['max'], stats['BBox_Area_px']['max']]
        }
        print(pd.DataFrame(output_data).to_markdown(index=False, floatfmt=".1f"))
        
    print("\n" + "="*50)
    print("統計完成。")

if __name__ == "__main__":
    analyze_simple_resolution()

In [ ]:
# 檢查 YOLO / ultralytics 版本，並清除舊的 .cache 和 .npy 檔案
import os
import glob
import ultralytics
from ultralytics.utils import DEFAULT_CFG_DICT

# 設定您的資料集路徑
dataset_dir = 'roboflow_1000_側拍_split'  # 請確認這是您的資料集根目錄

print(f"🔎 ultralytics 版本: {ultralytics.__version__}")

# 這份訓練程式碼目前使用到的參數名稱
used_params = [
    'data', 'epochs', 'batch', 'imgsz', 'device', 'workers', 'optimizer', 'lr0',
    'weight_decay', 'cos_lr', 'patience', 'box', 'cls', 'multi_scale', 'plots',
    'verbose', 'mosaic', 'mixup', 'copy_paste', 'scale', 'degrees', 'fliplr',
    'hsv_h', 'hsv_s', 'hsv_v', 'warmup_epochs', 'close_mosaic', 'freeze',
    'translate', 'perspective', 'shear', 'name'
]
unsupported = [p for p in used_params if p not in DEFAULT_CFG_DICT]
if unsupported:
    print(f"⚠️ 目前版本不支援的參數: {unsupported}")
else:
    print("✅ 目前使用的參數名稱都在本版 ultralytics 中支援")

print("📐 本次訓練將使用自訂架構: yolo11-strawberry-p2-cbam-s.yaml")
print("🔍 正在搜尋並刪除舊的 .cache 及 .npy 檔案...")

# 搜尋 train 和 val 資料夾下的 .cache 和 .npy
cache_files = glob.glob(os.path.join(dataset_dir, '**', '*.cache'), recursive=True) + glob.glob(os.path.join(dataset_dir, '**', '*.npy'), recursive=True)

if cache_files:
    for f in cache_files:
        try:
            os.remove(f)
            print(f"🗑️ 已刪除: {f}")
        except Exception as e:
            print(f"❌ 無法刪除 {f}: {e}")
    print("\n✅ 舊快取已清除，下次訓練時會重新掃描資料集。")
else:
    print("✅ 未發現舊的 .cache 檔案。")


In [ ]:
#yolo訓練程式碼 (一階段)
import os, torch, gc, sys, logging, warnings
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG_DICT
from agent_tools import error_logger, yolo_utils

# --- 🌟 初始化：環境優化與自定義模組 ---
yolo_utils.register_yolo_modules()
warnings.filterwarnings("ignore", message=".*deterministic.*")
warnings.filterwarnings("ignore", category=UserWarning)
try:
    torch.use_deterministic_algorithms(False)
except Exception:
    pass

# --- 🛡️ 強制恢復日誌顯示 ---
logging.getLogger("ultralytics").setLevel(logging.INFO)
os.environ["YOLO_VERBOSE"] = "True"

# --- 🛡️ 環境隔離 ---
sys.argv = [sys.argv[0]]


def validate_train_kwargs(train_kwargs, stage_name):
    unsupported = [name for name in train_kwargs if name not in DEFAULT_CFG_DICT]
    if unsupported:
        raise ValueError(f"{stage_name} 發現不支援的 YOLO 參數: {unsupported}")
    print(f"✅ {stage_name} 參數名稱全部符合目前 ultralytics 版本")


def count_cbam_modules(model):
    return sum(1 for module in model.model.modules() if module.__class__.__name__ == 'CBAM')


if __name__ == '__main__':
    # 1. 顯存與變數清理
    model = None
    if 'model' in globals() and model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        DEVICE_ID = 0
        print(f"🔥 使用 GPU: {torch.cuda.get_device_name(0)}")
    else:
        DEVICE_ID = 'cpu'

    # 2. 參數設定
    MODEL_CFG = "yolo11-strawberry-p2-cbam-s.yaml"
    PRETRAINED_WEIGHTS = "yolo11s.pt"
    DATA_YAML_PATH = "yolo_data/data.yaml"

    train_kwargs = {
        'data': DATA_YAML_PATH,
        'epochs': 300,
        'batch': 32,
        'imgsz': 640,
        'device': DEVICE_ID,
        'workers': 4,
        'optimizer': 'AdamW',
        'lr0': 0.001,
        'weight_decay': 0.001,
        'cos_lr': True,
        'patience': 30,
        'box': 15.0,
        'cls': 0.8,
        'multi_scale': False,
        'plots': True,
        'verbose': True,
        'mosaic': 1.0,
        'mixup': 0.2,
        'copy_paste': 0.02,
        'scale': 0.5,
        'degrees': 10.0,
        'fliplr': 0.5,
        'hsv_h': 0.015,
        'hsv_s': 0.7,
        'hsv_v': 0.4,
        'warmup_epochs': 3.0,
        'close_mosaic': 30,
    }

    try:
        validate_train_kwargs(train_kwargs, 'Stage 1')

        # 3. 建立模型：明確指定自訂架構 YAML
        model = YOLO(MODEL_CFG).load(PRETRAINED_WEIGHTS)
        print(f"\n🚀 啟動一階段訓練：{MODEL_CFG}")
        print(f"📐 Stage 1 採用的模型來源: {MODEL_CFG} + {PRETRAINED_WEIGHTS}")
        print(f"📐 Stage 1 目前 CBAM 模組數量: {count_cbam_modules(model)}")

        # 4. 啟動訓練
        model.train(**train_kwargs)
        print("\n✨ 一階段基礎訓練順利結束！")
    except Exception as e:
        error_logger.log_error(e, context="YOLO 一階段基礎訓練")
        raise


In [ ]:
# yolo訓練程式碼 (一階段：基礎訓練)
import os, torch, gc, sys, logging, warnings
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG_DICT
from agent_tools import error_logger, yolo_utils

yolo_utils.register_yolo_modules()
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("ultralytics").setLevel(logging.INFO)
sys.argv = [sys.argv[0]]

if __name__ == '__main__':
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    DEVICE_ID = 0 if torch.cuda.is_available() else 'cpu'

    # 🚀 使用升級後的 M 版本模型
    MODEL_CFG = "yolo11-strawberry-p2-cbam-m.yaml"
    PRETRAINED_WEIGHTS = "yolo11m.pt"
    DATA_YAML_PATH = "yolo_data/data.yaml"

    train_kwargs = {
        'data': DATA_YAML_PATH,
        'epochs': 300,
        'batch': 16,     # M 版本較大，batch 設為 16 較保險
        'imgsz': 640,    # 依照要求維持 640 避免 OOM
        'device': DEVICE_ID,
        'workers': 4,
        'optimizer': 'AdamW',
        'lr0': 0.001,
        'weight_decay': 0.001,
        'cos_lr': True,
        'patience': 100,
        'box': 7.5,
        'cls': 0.8,
        'mosaic': 1.0,
        'mixup': 0.1,
        'hsv_s': 0.5,
        'hsv_v': 0.4,
        'close_mosaic': 30,
    }

    model = YOLO(MODEL_CFG).load(PRETRAINED_WEIGHTS)
    model.train(**train_kwargs)


In [ ]:
#🎯 二階段：硬樣本微調 (Hard Example Fine-tuning)
import os, glob, torch, gc, shutil, yaml, datetime
from ultralytics import YOLO
from agent_tools import yolo_utils

yolo_utils.register_yolo_modules()

def prepare_hard_samples():
    """自動從最新診斷報告中提取有問題的圖片"""
    BASE_DIR = '診斷結果'
    SOURCE_IMG_DIR = 'labelImg-master/trainimg'
    HARD_DATA_DIR = 'yolo_hard_samples'
    
    # 1. 尋找最新的診斷報告
    report_dirs = glob.glob(os.path.join(BASE_DIR, '診斷_*'))
    if not report_dirs: return None
    latest_report_dir = max(report_dirs, key=os.path.getmtime)
    report_path = os.path.join(latest_report_dir, '全資料集診斷報告.txt')
    
    if not os.path.exists(report_path): return None
    
    # 2. 解析報告，提取 ISSUE 檔案名
    issue_files = []
    with open(report_path, 'r', encoding='utf-8') as f:
        for line in f:
            if '[ISSUE]' in line:
                filename = line.split('|')[0].replace('[ISSUE]', '').strip()
                issue_files.append(filename)
    
    if not issue_files:
        print("✅ 報告中沒有 ISSUE 圖片，跳過硬樣本微調。")
        return None

    print(f"🔥 偵測到 {len(issue_files)} 張硬樣本圖片，正在建立專屬資料集...")
    
    # 3. 建立硬樣本資料夾結構
    for sub in ['images', 'labels']: 
        p = os.path.join(HARD_DATA_DIR, sub)
        if os.path.exists(p): shutil.rmtree(p)
        os.makedirs(p)
    
    # 4. 複製圖片與標註
    for fname in issue_files:
        img_src = os.path.join(SOURCE_IMG_DIR, fname)
        lbl_src = os.path.join(SOURCE_IMG_DIR, os.path.splitext(fname)[0] + '.txt')
        if os.path.exists(img_src): shutil.copy(img_src, os.path.join(HARD_DATA_DIR, 'images', fname))
        if os.path.exists(lbl_src): shutil.copy(lbl_src, os.path.join(HARD_DATA_DIR, 'labels', os.path.basename(lbl_src)))
    
    # 5. 建立硬樣本 data.yaml
    hard_yaml = {
        'path': os.path.abspath(HARD_DATA_DIR),
        'train': 'images', 'val': 'images', # 硬樣本量少，直接用同一組做微調
        'nc': 3, 'names': ['level_1', 'level_2', 'level_3']
    }
    yaml_path = os.path.join(HARD_DATA_DIR, 'hard_samples.yaml')
    with open(yaml_path, 'w') as f: yaml.dump(hard_yaml, f)
    
    return yaml_path

if __name__ == '__main__':
    # 1. 尋找一階段最佳權重
    train_dirs = glob.glob(os.path.join("runs", "detect", "train*"))
    if not train_dirs: sys.exit("找不到一階段權重")
    latest_train = max(train_dirs, key=os.path.getmtime)
    best_weights = os.path.join(latest_train, "weights", "best.pt")

    # 2. 準備硬樣本
    hard_yaml_path = prepare_hard_samples()
    
    if hard_yaml_path and os.path.exists(best_weights):
        print(f"🚀 啟動二階段：針對硬樣本進行微調 (模型: {best_weights})")
        model = YOLO(best_weights)
        
        train_kwargs = {
            'data': hard_yaml_path,
            'epochs': 100,      # 延長訓練時間
            'batch': 16,
            'imgsz': 640,
            'optimizer': 'AdamW',
            'lr0': 0.0001,      # 極低學習率微調
            'freeze': 10,       # 凍結骨幹，專注於 Head 調整
            'patience': 0,      # 二階段微調不建議早停，確保完整訓練
            'name': os.path.basename(latest_train) + '_hard_fine_tuned'
        }
        
        model.train(**train_kwargs)
        print("✨ 硬樣本微調結束！")
